# HW2 — Feature Engineering: Irrigation Need Prediction (S6E4)
**Tyler Wolf Williams**

This notebook engineers new features on top of the raw Kaggle dataset, evaluates them using LightGBM cross-validation, explains them with SHAP, and saves the best feature set for the models notebook.

**Approach overview:**
- Numerical: log transforms, evapotranspiration proxy, water balance, soil quality index, pairwise interactions
- Categorical: cross-features (crop × stage, season × region), crop-season alignment
- Statistical: cross-validated target encoding, group aggregations
- Leakage investigation: test with/without `Irrigation_Type` and `Water_Source`
- Feature evaluation: SHAP importance + LightGBM CV

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (11, 5)

train = pd.read_csv('../Homework 2/train.csv')
test  = pd.read_csv('../Homework 2/test.csv')

TARGET = 'Irrigation_Need'
ID_COL = 'id'

label_map     = {'Low': 0, 'Medium': 1, 'High': 2}
inv_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
train['target'] = train[TARGET].map(label_map)

cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
num_cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
            'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
            'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

print(f'Train: {train.shape}  Test: {test.shape}')
print('\nTarget distribution:')
print(train[TARGET].value_counts())
print(train[TARGET].value_counts(normalize=True).round(4))

Train: (630000, 22)  Test: (270000, 20)

Target distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


## 1. Numerical Feature Engineering

**Domain motivation:** Irrigation need is driven by evapotranspiration (ET) minus water supply. The Penman-Monteith equation shows ET increases with temperature, solar radiation, and wind speed, and decreases with humidity. We approximate this and derive complementary features.

In [2]:
def add_numerical_features(df):
    df = df.copy()

    # --- Log transforms: compress right-skewed distributions ---
    for col in ['Rainfall_mm', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']:
        df[f'log_{col}'] = np.log1p(df[col])

    # --- Evapotranspiration proxy (simplified Penman-Monteith) ---
    # ET increases with temp, sunlight, wind; decreases with humidity
    df['ET_proxy'] = (
        df['Temperature_C'] * df['Sunlight_Hours'] * df['Wind_Speed_kmh']
    ) / (df['Humidity'] + 1)

    # --- Water balance: rainfall + soil moisture minus ET demand ---
    # Negative = deficit (irrigation needed); positive = surplus
    df['water_balance'] = df['Rainfall_mm'] + df['Soil_Moisture'] - df['ET_proxy']

    # --- Aridity index: rainfall per degree of temperature ---
    df['aridity_index'] = df['Rainfall_mm'] / (df['Temperature_C'] + 1)

    # --- Soil quality composite ---
    # Good soil: pH near neutral (6.5), high organic carbon, low salinity (EC)
    df['pH_deviation'] = np.abs(df['Soil_pH'] - 6.5)
    df['soil_quality']  = (
        df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 0.1) - df['pH_deviation']
    )

    # --- Key pairwise interactions ---
    df['moisture_x_temp']  = df['Soil_Moisture'] * df['Temperature_C']   # heat-drought stress
    df['rain_x_wind']      = df['Rainfall_mm']   * df['Wind_Speed_kmh']  # wind-driven evaporation
    df['pH_x_EC']          = df['Soil_pH']        * df['Electrical_Conductivity']  # soil chemistry
    df['humidity_x_temp']  = df['Humidity']       * df['Temperature_C']  # apparent humidity

    # --- Irrigation intensity: previous irrigation per hectare ---
    df['prev_irr_per_ha'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 0.01)

    return df

train = add_numerical_features(train)
test  = add_numerical_features(test)

new_num_feats = [
    'log_Rainfall_mm', 'log_Wind_Speed_kmh', 'log_Field_Area_hectare', 'log_Previous_Irrigation_mm',
    'ET_proxy', 'water_balance', 'aridity_index', 'pH_deviation', 'soil_quality',
    'moisture_x_temp', 'rain_x_wind', 'pH_x_EC', 'humidity_x_temp', 'prev_irr_per_ha'
]
print(f'Added {len(new_num_feats)} numerical features.')
print(train[new_num_feats].describe().round(3))

Added 14 numerical features.
       log_Rainfall_mm  log_Wind_Speed_kmh  log_Field_Area_hectare  \
count       630000.000          630000.000              630000.000   
mean             7.150               2.258                   1.979   
std              0.655               0.650                   0.626   
min              0.322               0.405                   0.262   
25%              6.862               1.837                   1.585   
50%              7.292               2.441                   2.126   
75%              7.628               2.799                   2.497   
max              7.824               3.045                   2.773   

       log_Previous_Irrigation_mm    ET_proxy  water_balance  aridity_index  \
count                  630000.000  630000.000     630000.000     630000.000   
mean                        3.916      37.858       1461.654         58.397   
std                         0.804      32.934        615.891         33.015   
min                     

## 2. Categorical Feature Engineering

Cross-features capture interaction effects that neither feature alone expresses. For example, `Crop_Type` and `Crop_Growth_Stage` together determine water requirements better than either individually.

In [3]:
print('Crop types in dataset:', sorted(train['Crop_Type'].unique()))
print('Seasons:               ', sorted(train['Season'].unique()))
print('Crop growth stages:    ', sorted(train['Crop_Growth_Stage'].unique()))
print('Regions:               ', sorted(train['Region'].unique()))
print('Soil types:            ', sorted(train['Soil_Type'].unique()))

Crop types in dataset: ['Cotton', 'Maize', 'Potato', 'Rice', 'Sugarcane', 'Wheat']
Seasons:                ['Kharif', 'Rabi', 'Zaid']
Crop growth stages:     ['Flowering', 'Harvest', 'Sowing', 'Vegetative']
Regions:                ['Central', 'East', 'North', 'South', 'West']
Soil types:             ['Clay', 'Loamy', 'Sandy', 'Silt']


In [4]:
for df in [train, test]:
    # Categorical cross-features (Cartesian combinations)
    df['crop_stage']    = df['Crop_Type'].astype(str) + '_' + df['Crop_Growth_Stage'].astype(str)
    df['season_region'] = df['Season'].astype(str)    + '_' + df['Region'].astype(str)
    df['soil_crop']     = df['Soil_Type'].astype(str) + '_' + df['Crop_Type'].astype(str)

new_cat_feats = ['crop_stage', 'season_region', 'soil_crop']
print(f'crop_stage unique values:    {train["crop_stage"].nunique()}')
print(f'season_region unique values: {train["season_region"].nunique()}')
print(f'soil_crop unique values:     {train["soil_crop"].nunique()}')

crop_stage unique values:    24
season_region unique values: 15
soil_crop unique values:     24


## 3. Target Encoding (Cross-Validated, 5-Fold)

Target encoding replaces a categorical value with the mean target label for that category. Cross-validation prevents leakage: each sample's encoding is computed from the **other 4 folds**, not from itself.

In [5]:
def cv_target_encode(train_df, test_df, cat_col, target_col='target', n_splits=5):
    """5-fold cross-validated target encoding (prevents leakage)."""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_encoded = np.zeros(len(train_df))
    global_mean   = train_df[target_col].mean()

    for tr_idx, val_idx in kf.split(train_df, train_df[target_col]):
        fold_means = train_df.iloc[tr_idx].groupby(cat_col)[target_col].mean()
        train_encoded[val_idx] = (
            train_df.iloc[val_idx][cat_col].map(fold_means).fillna(global_mean)
        )

    test_means    = train_df.groupby(cat_col)[target_col].mean()
    test_encoded  = test_df[cat_col].map(test_means).fillna(global_mean)
    return train_encoded, test_encoded

te_cols = ['Crop_Type', 'Soil_Type', 'Season', 'Region', 'crop_stage', 'season_region', 'soil_crop']

for col in te_cols:
    tr_enc, te_enc   = cv_target_encode(train, test, col)
    train[f'te_{col}'] = tr_enc
    test[f'te_{col}']  = te_enc

te_feat_names = [f'te_{c}' for c in te_cols]
print(f'Target-encoded {len(te_cols)} categorical features.')

corr = (train[te_feat_names + ['target']]
        .corr()['target'].drop('target').abs().sort_values(ascending=False))
print('\n|Correlation| of TE features with ordinal target:')
print(corr.round(4))

Target-encoded 7 categorical features.

|Correlation| of TE features with ordinal target:
te_crop_stage       0.5418
te_soil_crop        0.0337
te_Crop_Type        0.0264
te_season_region    0.0264
te_Season           0.0233
te_Soil_Type        0.0210
te_Region           0.0130
Name: target, dtype: float64


## 4. Group Statistics

Within-group statistics encode the typical soil moisture, temperature, and rainfall for each crop type and region — capturing population-level patterns without target leakage.

In [6]:
def add_group_stats(train_df, test_df, group_col, stat_col):
    stats = train_df.groupby(group_col)[stat_col].agg(['mean', 'std']).fillna(0)
    stats.columns = [f'{group_col}_{stat_col}_mean', f'{group_col}_{stat_col}_std']
    return train_df.join(stats, on=group_col), test_df.join(stats, on=group_col)

group_pairs = [
    ('Crop_Type', 'Soil_Moisture'),
    ('Region',    'Soil_Moisture'),
    ('Season',    'Temperature_C'),
    ('Region',    'Rainfall_mm'),
    ('Crop_Type', 'ET_proxy'),
]
for g, s in group_pairs:
    train, test = add_group_stats(train, test, g, s)

grp_feat_names = [f'{g}_{s}_{agg}' for g, s in group_pairs for agg in ['mean', 'std']]
print(f'Added {len(grp_feat_names)} group statistic features.')

Added 10 group statistic features.


## 5. Leakage Investigation

`Irrigation_Type` and `Water_Source` may be **downstream** of the target — the type of irrigation system used might be selected *after* the irrigation need is assessed. We test whether they drive model performance or just add noise.

In [7]:
# Label-encode all categoricals before modeling
all_cat_cols = cat_cols + new_cat_feats
for col in all_cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

base_features = num_cols + cat_cols
all_features  = list(dict.fromkeys(
    base_features + new_num_feats + new_cat_feats + te_feat_names + grp_feat_names
))

# Features without the suspected leakage columns
no_leak_features = [f for f in all_features
                    if 'Irrigation_Type' not in f and 'Water_Source' not in f]

print(f'Base features:            {len(base_features)}')
print(f'All features:             {len(all_features)}')
print(f'Without leakage columns:  {len(no_leak_features)}')

def quick_lgb_cv(X, y, n_splits=5, label=''):
    kf_cv  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    model  = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.1, num_leaves=63,
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
    )
    acc = cross_val_score(model, X, y, cv=kf_cv, scoring='accuracy', n_jobs=-1)
    f1  = cross_val_score(model, X, y, cv=kf_cv, scoring='f1_macro',  n_jobs=-1)
    print(f'{label:<40} Acc={acc.mean():.4f}±{acc.std():.4f}  F1={f1.mean():.4f}±{f1.std():.4f}')
    return acc.mean(), f1.mean()

y = train['target']

print('\n5-fold CV evaluation (LightGBM, class_weight=balanced):\n')
acc_base, f1_base = quick_lgb_cv(train[base_features],    y, label='Base features (19)')
acc_all,  f1_all  = quick_lgb_cv(train[all_features],     y, label='All features (incl. leakage cols)')
acc_nl,   f1_nl   = quick_lgb_cv(train[no_leak_features], y, label='All features (excl. leakage cols)')

leak_impact_acc = acc_all - acc_nl
leak_impact_f1  = f1_all  - f1_nl
print(f'\nLeakage columns impact: ΔAcc={leak_impact_acc:+.4f}  ΔF1={leak_impact_f1:+.4f}')
if abs(leak_impact_acc) > 0.02:
    print('NOTE: Large impact — these features are highly correlated with target (possible leakage).')
else:
    print('NOTE: Small impact — leakage columns are not dominating predictions.')

Base features:            19
All features:             53
Without leakage columns:  51

5-fold CV evaluation (LightGBM, class_weight=balanced):

Base features (19)                       Acc=0.9838±0.0003  F1=0.9677±0.0008
All features (incl. leakage cols)        Acc=0.9841±0.0003  F1=0.9684±0.0010
All features (excl. leakage cols)        Acc=0.9839±0.0004  F1=0.9680±0.0015

Leakage columns impact: ΔAcc=+0.0001  ΔF1=+0.0003
NOTE: Small impact — leakage columns are not dominating predictions.


## 6. SHAP Feature Importance

In [ ]:
# Train SHAP model on a 20K stratified sample (SHAP is O(n²) — full data is too slow)
sample      = train.sample(20000, random_state=42)
X_shap_df   = sample[all_features]
y_shap      = sample['target']

lgb_shap = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.1, num_leaves=63,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)
lgb_shap.fit(X_shap_df, y_shap)

explainer  = shap.TreeExplainer(lgb_shap)
shap_vals  = explainer.shap_values(X_shap_df)

# Mean |SHAP| across all 3 classes → single importance score per feature.
# Newer SHAP returns a 3D array (n_samples, n_features, n_classes) instead of a list;
# averaging over axis=0 alone gives (n_features, n_classes) which is 2D and breaks DataFrame.
shap_arr = np.array(shap_vals)
if shap_arr.ndim == 3:
    # New SHAP format: (n_samples, n_features, n_classes) — average over samples AND classes
    mean_shap = np.abs(shap_arr).mean(axis=(0, 2))
elif isinstance(shap_vals, list):
    # Old SHAP format: list of (n_samples, n_features) arrays, one per class
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
else:
    # Binary: (n_samples, n_features)
    mean_shap = np.abs(shap_arr).mean(axis=0)

mean_shap = np.array(mean_shap).ravel()  # guarantee 1D regardless of SHAP version

shap_df = (
    pd.DataFrame({'feature': all_features, 'mean_abs_shap': mean_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
print('Top 30 features by mean |SHAP| (averaged across all 3 classes):')
print(shap_df.head(30).to_string(index=False))

In [ ]:
# Bar chart of top 20
fig, ax = plt.subplots(figsize=(10, 9))
top20 = shap_df.head(20)
colors = ['salmon' if any(tok in f for tok in ['Irrigation_Type', 'Water_Source'])
          else 'steelblue' for f in top20['feature'].values[::-1]]
ax.barh(range(20), top20['mean_abs_shap'].values[::-1], color=colors, alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels(top20['feature'].values[::-1])
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Top 20 Features — Mean Absolute SHAP\n(LightGBM on 20K sample; red = potential leakage)', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../Homework 2/shap_importance_hw2.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP beeswarm for 'High' class (minority — most interesting to explain)
# Handle both new 3D array format and old list format
high_class_idx = 2  # 0=Low, 1=Medium, 2=High
if shap_arr.ndim == 3:
    shap_for_plot = shap_arr[:, :, high_class_idx]   # (n_samples, n_features)
elif isinstance(shap_vals, list):
    shap_for_plot = shap_vals[high_class_idx]
else:
    shap_for_plot = shap_arr

shap.summary_plot(
    shap_for_plot, X_shap_df,
    feature_names=all_features, max_display=15, plot_type='dot', show=False
)
plt.title("SHAP Values — 'High' Irrigation Class (Top 15 Features)", fontsize=12)
plt.tight_layout()
plt.savefig('../Homework 2/shap_beeswarm_hw2.png', dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plots saved.')

## 7. Feature Selection & Save

In [ ]:
# Keep features whose mean |SHAP| exceeds the 25th percentile (drop bottom quarter)
threshold       = shap_df['mean_abs_shap'].quantile(0.25)
selected_feats  = shap_df[shap_df['mean_abs_shap'] > threshold]['feature'].tolist()
dropped_feats   = [f for f in all_features if f not in selected_feats]

print(f'Kept {len(selected_feats)} / {len(all_features)} features  '
      f'(dropped {len(dropped_feats)} low-importance features)')
print('\nDropped features (bottom 25% SHAP):')
print(dropped_feats)

print('\nFinal 5-fold CV with selected features:')
acc_sel, f1_sel = quick_lgb_cv(train[selected_feats], y, label='Selected features')

print(f"""
{'─'*65}
{'Feature Set':<40} {'Acc':>7}  {'F1':>7}
{'─'*65}
{'Base (19 features)':<40} {acc_base:>7.4f}  {f1_base:>7.4f}
{'All engineered (incl. leakage cols)':<40} {acc_all:>7.4f}  {f1_all:>7.4f}
{'All engineered (excl. leakage cols)':<40} {acc_nl:>7.4f}  {f1_nl:>7.4f}
{'Selected (top SHAP)':<40} {acc_sel:>7.4f}  {f1_sel:>7.4f}
{'─'*65}
""")

In [ ]:
# Save processed datasets for the models notebook
train_save = train[selected_feats + ['target', TARGET, ID_COL]]
test_save  = test[selected_feats + [ID_COL]]

train_save.to_csv('../Homework 2/train_engineered.csv', index=False)
test_save.to_csv('../Homework 2/test_engineered.csv',   index=False)

with open('../Homework 2/selected_features.txt', 'w') as fh:
    fh.write('\n'.join(selected_feats))

print('Saved:')
print('  ../Homework 2/train_engineered.csv')
print('  ../Homework 2/test_engineered.csv')
print('  ../Homework 2/selected_features.txt')
print(f'\nFinal feature set has {len(selected_feats)} features.')